# Optical Simulation
This script do Jones calculus simulation for oke experiments

## import

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from dataclasses import dataclass
from tqdm import tqdm

## class and functions

In [ ]:
@dataclass
class beam:
    ex: complex
    ey: complex

    @classmethod
    def make_beam_from_amplitude_phase(cls, ex_amp: float, ex_phase: float, ey_amp: float, ey_phase: float):
        ex = ex_amp * np.exp(1j * ex_phase)
        ey = ey_amp * np.exp(1j * ey_phase)
        return cls(ex, ey)
    
    @classmethod
    def make_linearly_polarized_source(cls, amp, angle):
        ex_amp = amp * np.cos(angle)
        ey_amp = amp * np.sin(angle)
        ex_phase = 0
        ey_phase = 0
        return cls.make_beam_from_amplitude_phase(ex_amp, ex_phase, ey_amp, ey_phase)

    def __add__(self, other):
        if not isinstance(other, beam):
            return NotImplemented
        ex = self.ex + other.ex
        ey = self.ey + other.ey
        return beam(ex, ey)

    def __sub__(self, other):
        if not isinstance(other, beam):
            return NotImplemented
        ex = self.ex - other.ex
        ey = self.ey - other.ey
        return beam(ex, ey)

    def times_matrix(self, mat: np.ndarray):
        if mat.shape != (2, 2):
            raise ValueError("Matrix must be 2x2")
        vec = np.array([self.ex, self.ey])
        result = mat @ vec
        ex_new, ey_new = result[0], result[1]
        return beam(ex_new, ey_new)
    
    # optical element from Jones calculus
    def apply_polarizer(self, angle: float):
        """
        Apply a polarizer at the given angle (in radians) to the beam.
        """
        c = np.cos(angle)
        s = np.sin(angle)
        mat = np.array([[c**2, c*s],
                        [c*s, s**2]])
        return self.times_matrix(mat)
    
    def apply_waveplate(self, angle: float, phase_shift: float):
        """
        Apply a waveplate at the given angle (in radians) with the specified phase shift to the beam.
        """
        c = np.cos(angle)
        s = np.sin(angle)
        mat = np.array([[c**2 + s**2 * np.exp(1j * phase_shift), c*s * (1 - np.exp(1j * phase_shift))],
                        [c*s * (1 - np.exp(1j * phase_shift)), s**2 + c**2 * np.exp(1j * phase_shift)]])
        return self.times_matrix(mat)
    
    # shortcuts for hwp and qwp
    def apply_hwp(self, angle: float):
        """
        Apply a half-wave plate at the given angle (in radians) to the beam.
        """
        return self.apply_waveplate(angle, np.pi)

    def apply_qwp(self, angle: float):
        """
        Apply a quarter-wave plate at the given angle (in radians) to the beam.
        """
        return self.apply_waveplate(angle, np.pi / 2)
    
    def apply_nd_filter(self, OD: float):
        """
        Apply a neutral density (ND) filter with the specified optical density (OD) to the beam.
        The optical density should be greater than 0.
        """
        if not OD > 0:
            raise ValueError("Optical density must be greater than 0")
        attenuation = 10**(-OD / 2)
        return beam(self.ex * attenuation, self.ey * attenuation)

    def apply_wollaston(self, angle: float):
        """
        Apply a Wollaston prism at the given angle (in radians) to the beam.
        """
        angle1 = angle
        angle2 = angle + np.pi / 2
        beam1 = self.apply_polarizer(angle1)
        beam2 = self.apply_polarizer(angle2)
        return beam1, beam2

    def apply_pem(self, frequency:float, modulation:float, t:float):
        """
        Apply a photoelastic modulator (PEM) to the beam. The optical axis of PEM is set to be 0 degree.

        frequency: modulation frequency of the PEM (in Hz)
        modulation: modulation depth of the PEM (in radians)
        t: time at which to evaluate the modulation
        """
        phase = modulation * np.sin(2 * np.pi * frequency * t)
        return self.apply_waveplate(0, phase)

    # other physical processes:
    def intensity(self):
        """
        Calculate the intensity of the beam.
        """
        return np.abs(self.ex)**2 + np.abs(self.ey)**2

    # other utilities
    def seperate_xy_beam(self):
        """
        Separate the beam into its x and y components as individual beams.
        """
        beam_x = beam(self.ex, 0)
        beam_y = beam(0, self.ey)
        return beam_x, beam_y

    def string_details(self):
        string = f"total beam intensity: {self.intensity()}\nx component intensity: {np.abs(self.ex)**2}\ny component intensity: {np.abs(self.ey)**2}\nxy_phase_difference: {np.angle(self.ex) - np.angle(self.ey)}"
        return string
    
    # visualization
    def plot_polarization_ellipse(self):
        """
        Plot the polarization ellipse of the beam.
        """
        t = np.linspace(0, 2 * np.pi, 1000)
        x = np.real(self.ex * np.exp(1j * t))
        y = np.real(self.ey * np.exp(1j * t))
        fig, ax = plt.subplots()
        ax.plot(x, y)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title("Polarization Ellipse")
        ax.axis("equal")
        return fig, ax
    

## optical setup for calibration

In [ ]:
# # parameters
# wavelength = 658e-9 # in meters
# t = np.arange(0, 0.5e-4, 1E-7)  # time array from 0 to 1 second 1 microsecond resolution.

# amp_noise = 0.010  # relative noise level for the light source
# hwp_pol_err = -0  # incident hwp error in angle
# pem_freq = 50 * 1e3  # frequency of the PEM in Hz
# pem_misphase_noise = 0.01 # misphase noise for the PEM
# pem_modulation = 180  # modulation  for the PEM in degree
# pem_modulation_err = 0 # absolution error for the PEM modulation in degree

# # for testing, use 2nd polarizer
# analyzer_pol = 135  # polarization angle of the analyzer polarizer in degree
# analyzer_pol_err = 0  # absolute error for the analyzer polarizer angle

## simulation calibration

In [ ]:
# intensity = np.zeros(len(t))
# for i in tqdm(range(len(t))):
#     # light sources
#     l = beam.make_linearly_polarized_source(1.0+amp_noise*np.random.randn(), 0)   # horizontally polarized light with amplitude 1.0
#     # apply hwp to 45 deg
#     l = l.apply_hwp((22.5 + hwp_pol_err) * np.pi / 180)
#     # go through horizonal placed PEM
#     pem_mistime = pem_misphase_noise * np.random.randn() / pem_freq
#     l = l.apply_pem(pem_freq, (pem_modulation + pem_modulation_err) * np.pi / 180, t[i]+pem_mistime)
#     # apply second crossed polarizer
#     l = l.apply_polarizer((analyzer_pol + analyzer_pol_err) * np.pi / 180)  # crossed polarizer at 90 degrees
#     # analyze the resulting beam
#     intensity[i] = l.intensity()


## visualization

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, intensity, label = 'raw')
ax.legend()
ax.set_xlabel('Time (s)')
ax.set_ylabel('Intensity')
ax.set_title('Intensity vs Time')
plt.show()